In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import sklearn
import pathlib
import os

In [ ]:
from pathlib import Path

positive_directory = Path('data/Positive')
negative_directory = Path('data/Negative')
uncertain_directory = Path('data/Uncertain')

positive_labels_file = "data/Positive/Positive.xlsx"
negative_labels_file = "data/Negative/Negative.xlsx"
uncertain_labels_file = "data/Uncertain/Uncertain.xlsx"

In [ ]:
from skimage.feature import graycomatrix, graycoprops
# label_map = {'A': 'Positive', 'B': 'Positive', 'C': 'Negative', 'D': 'Uncertain', 'E': 'Uncertain'}  # <-- Remove this for training

X = []
y = []

In [ ]:
# process positive files

positive_labels_df = pd.read_excel(positive_labels_file)
positive_label_dict = dict(zip(positive_labels_df['fileName'], positive_labels_df['Label']))  # Use original letter label

for file in positive_directory.glob('*.jpg'):
    if file.is_file():
        img = cv2.imread(str(file))
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        glcm = graycomatrix(img_gray, distances=[5], angles=[0], levels=256, symmetric=True, normed=True)
        contrast = graycoprops(glcm, 'contrast')[0, 0]
        dissimilarity = graycoprops(glcm, 'dissimilarity')[0, 0]
        homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
        energy = graycoprops(glcm, 'energy')[0, 0]
        correlation = graycoprops(glcm, 'correlation')[0, 0]

        features = [contrast, dissimilarity, homogeneity, energy, correlation]
        X.append(features)
        image_label = positive_label_dict.get(file.name)
        y.append(image_label)

In [ ]:
# process negative files

negative_labels_df = pd.read_excel(negative_labels_file)
negative_label_dict = dict(zip(negative_labels_df['fileName'], negative_labels_df['Label']))  # Use original letter label

for file in negative_directory.glob('*.jpg'):
    if file.is_file():
        img = cv2.imread(str(file))
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        glcm = graycomatrix(img_gray, distances=[5], angles=[0], levels=256, symmetric=True, normed=True)
        contrast = graycoprops(glcm, 'contrast')[0, 0]
        dissimilarity = graycoprops(glcm, 'dissimilarity')[0, 0]
        homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
        energy = graycoprops(glcm, 'energy')[0, 0]
        correlation = graycoprops(glcm, 'correlation')[0, 0]

        features = [contrast, dissimilarity, homogeneity, energy, correlation]
        X.append(features)
        image_label = negative_label_dict.get(file.name)
        y.append(image_label)


In [ ]:
# process uncertain files

uncertain_labels_df = pd.read_excel(uncertain_labels_file)
uncertain_label_dict = dict(zip(uncertain_labels_df['fileName'], uncertain_labels_df['Label']))  # Use original letter label

for file in uncertain_directory.glob('*.jpg'):
    if file.is_file():
        img = cv2.imread(str(file))
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        glcm = graycomatrix(img_gray, distances=[5], angles=[0], levels=256, symmetric=True, normed=True)
        contrast = graycoprops(glcm, 'contrast')[0, 0]
        dissimilarity = graycoprops(glcm, 'dissimilarity')[0, 0]
        homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
        energy = graycoprops(glcm, 'energy')[0, 0]
        correlation = graycoprops(glcm, 'correlation')[0, 0]

        features = [contrast, dissimilarity, homogeneity, energy, correlation]
        X.append(features)
        image_label = uncertain_label_dict.get(file.name)
        y.append(image_label)


In [ ]:
X = np.array(X)
y = np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier()
classifier.fit(X_train, y_train)

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(classifier, X_test, y_test, cv=5)
print('Cross-validation scores:', scores)
print('Mean accuracy:', np.mean(scores))

In [ ]:
def test_from_image(filepath):
    # give it filepath to an image and have it predict the letter classification
    img = cv2.imread(str(filepath))
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    glcm = graycomatrix(img_gray, distances=[5], angles=[0], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    dissimilarity = graycoprops(glcm, 'dissimilarity')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]
    correlation = graycoprops(glcm, 'correlation')[0, 0]

    features = [contrast, dissimilarity, homogeneity, energy, correlation]
    features = np.array(features).reshape(1, -1)

    predicted_label = classifier.predict(features)[0]
    print(f'File: {filepath}, Predicted Label: {predicted_label}')
    return predicted_label
